# Reading Difficulty Predictor — RoBERTa (Final Dataset)
Upload `final_train.csv`, `final_val.csv`, `final_test.csv` to `/content` before running.

In [ ]:
# ============================================================
#  INSTALL
# ============================================================
!pip install -q transformers scikit-learn

# ============================================================
#  IMPORTS
# ============================================================
import os, torch
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizerFast,
    RobertaModel,
    AdamW,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import accuracy_score, classification_report

DATA_DIR   = '/content'
SAVE_PATH  = '/content/best_roberta_final.pt'
EPOCHS     = 5
BATCH_SIZE = 16
MAX_LEN    = 256
LR_BERT    = 1e-5
LR_HEAD    = 1e-3
PATIENCE   = 2
NUMERIC_COLS = [
    'flesch_kincaid_grade', 'word_count', 'sentence_count',
    'avg_word_length', 'avg_sentence_length', 'syllable_count',
    'unique_word_ratio', 'long_word_ratio',
    'flesch_reading_ease', 'difficult_word_percentage'
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
#  LOAD DATA
# ============================================================
train_df = pd.read_csv(os.path.join(DATA_DIR, 'final_train.csv')).dropna(subset=['text','label'])
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'final_val.csv')).dropna(subset=['text','label'])
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'final_test.csv')).dropna(subset=['text','label'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Label distribution (train):\n', train_df['label'].value_counts().sort_index())

# ============================================================
#  DATASET CLASS
# ============================================================
tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base')

class ReadingDataset(Dataset):
    def __init__(self, df):
        self.texts    = df['text'].astype(str).tolist()
        self.numerics = df[NUMERIC_COLS].fillna(0).values
        self.labels   = df['label'].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'numeric_feats':  torch.tensor(self.numerics[idx], dtype=torch.float),
            'labels':         torch.tensor(self.labels[idx],   dtype=torch.long)
        }

train_loader = DataLoader(ReadingDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(ReadingDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ReadingDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ============================================================
#  MODEL  (RoBERTa pooler + 10 numeric features → MLP → 3 classes)
# ============================================================
class RobertaHybrid(nn.Module):
    def __init__(self, num_labels=3, n_numeric=10):
        super().__init__()
        self.roberta    = RobertaModel.from_pretrained('roberta-base')
        self.dropout    = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(768 + n_numeric, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_labels)
        )

    def forward(self, input_ids, attention_mask, numeric_feats):
        out      = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled   = self.dropout(out.pooler_output)   # [CLS] through a linear+tanh
        combined = torch.cat([pooled, numeric_feats], dim=1)
        return self.classifier(combined)

model = RobertaHybrid().to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

# ============================================================
#  OPTIMIZER + SCHEDULER
# ============================================================
optimizer = AdamW([
    {'params': model.roberta.parameters(),    'lr': LR_BERT},
    {'params': model.classifier.parameters(), 'lr': LR_HEAD}
], weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.06)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
loss_fn      = nn.CrossEntropyLoss()

# ============================================================
#  TRAINING LOOP
# ============================================================
best_val_acc     = 0.0
patience_counter = 0

print('\n' + '='*55)
print('        STARTING RoBERTa TRAINING')
print('='*55)

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        logits = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device),
            batch['numeric_feats'].to(device)
        )
        loss = loss_fn(logits, batch['labels'].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if step % 200 == 0 and step > 0:
            print(f'  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_train_loss = total_loss / len(train_loader)

    # --- Validate ---
    model.eval()
    val_loss, preds, truths = 0, [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device),
                batch['numeric_feats'].to(device)
            )
            val_loss += loss_fn(logits, batch['labels'].to(device)).item()
            preds  += torch.argmax(logits, dim=1).cpu().tolist()
            truths += batch['labels'].tolist()

    val_acc      = accuracy_score(truths, preds)
    avg_val_loss = val_loss / len(val_loader)

    print(f'\n--- Epoch {epoch+1}/{EPOCHS} ---')
    print(f'  Train Loss : {avg_train_loss:.4f}')
    print(f'  Val   Loss : {avg_val_loss:.4f}')
    print(f'  Val   Acc  : {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  --> New best! Saved. (Best Val Acc: {best_val_acc:.4f})\n')
    else:
        patience_counter += 1
        print(f'  --> No improvement. Patience: {patience_counter}/{PATIENCE}\n')
        if patience_counter >= PATIENCE:
            print('  Early stopping triggered.')
            break

# ============================================================
#  FINAL TEST EVALUATION
# ============================================================
print('\nLoading best model for final evaluation...')
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

test_preds, test_truths = [], []
with torch.no_grad():
    for batch in test_loader:
        logits = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device),
            batch['numeric_feats'].to(device)
        )
        test_preds  += torch.argmax(logits, dim=1).cpu().tolist()
        test_truths += batch['labels'].tolist()

test_acc = accuracy_score(test_truths, test_preds)

print('\n' + '='*55)
print('   FINAL TEST RESULTS — RoBERTa (FINAL DATASET)')
print('='*55)
print(f'  Accuracy : {test_acc:.4f}')
print()
print(classification_report(test_truths, test_preds,
      target_names=['Easy (0)', 'Medium (1)', 'Hard (2)']))
print('='*55)
print(f'Model saved at: {SAVE_PATH}')